# Module 01: NumPy for Machine Learning
## Notebook 05: Practical Machine Learning Algorithms from Scratch

The most rigorous way to understand Machine Learning algorithms is to implement them directly from mathematical first principles using pure NumPy. In this capstone notebook, we progress from basic preprocessing to gradient descent optimizers, momentum mechanics, and multi-class neural output layers.

---

### Learning Objectives
By the end of this notebook, you will be able to:
1. Build reusable feature transformers (`StandardScaler`) adhering to Scikit-Learn's `.fit()` / `.transform()` paradigm.
2. Compute **vectorized pairwise distance matrices** without loops using quadratic expansion.
3. Construct an efficient **Mini-Batch Data Generator** for stochastic optimization.
4. Implement **Linear Regression with Batch Gradient Descent** from scratch and track loss convergence.
5. Implement **Binary Logistic Regression** with cross-entropy loss and compute evaluation metrics.
6. Implement **Ridge Regression with Closed-Form Regularization** exempting the bias term.
7. Implement **Mini-Batch SGD with Momentum** to accelerate convergence.
8. Build a complete **Multi-Class Softmax Regression** classifier from scratch.

In [10]:
import numpy as np
import time

print(f"NumPy version: {np.__version__}")

NumPy version: 2.5.3


### 1. Feature Preprocessing: `StandardScaler` from Scratch

In machine learning, gradient-based optimizers and distance-based algorithms are sensitive to feature scales.
A proper transformer must:
1. Compute $\mu$ and $\sigma$ strictly on **training data** during `.fit()` to prevent **data leakage**.
2. Apply the learned $\mu$ and $\sigma$ to both training and test data during `.transform()`.
3. Handle zero standard deviation safely with numerical tolerance $\epsilon = 10^{-8}$.

In [11]:
class StandardScalerFromScratch:
    # Standardizes features by removing the mean and scaling to unit variance
    def __init__(self, eps=1e-8):
        self.eps = eps
        self.mean_ = None
        self.scale_ = None
        
    def fit(self, X):
        # Compute mean and standard deviation along axis 0 (features)
        self.mean_ = np.mean(X, axis=0, keepdims=True)
        self.scale_ = np.std(X, axis=0, keepdims=True)
        # Prevent division by zero if a feature is constant
        self.scale_ = np.where(self.scale_ < self.eps, 1.0, self.scale_)
        return self
        
    def transform(self, X):
        if self.mean_ is None or self.scale_ is None:
            raise RuntimeError("Transformer has not been fitted yet!")
        return (X - self.mean_) / self.scale_
        
    def fit_transform(self, X):
        return self.fit(X).transform(X)
        
    def inverse_transform(self, X_scaled):
        return (X_scaled * self.scale_) + self.mean_

# Test the scaler
X_train = np.array([
    [10.0, 50000.0],
    [20.0, 70000.0],
    [30.0, 90000.0]
])

scaler = StandardScalerFromScratch()
X_scaled = scaler.fit_transform(X_train)

print("Original Features:\n", X_train)
print("\nScaled Features (zero mean, unit variance):\n", np.round(X_scaled, 4))
print("\nInverted Back to Original:\n", scaler.inverse_transform(X_scaled))

Original Features:
 [[1.e+01 5.e+04]
 [2.e+01 7.e+04]
 [3.e+01 9.e+04]]

Scaled Features (zero mean, unit variance):
 [[-1.2247 -1.2247]
 [ 0.      0.    ]
 [ 1.2247  1.2247]]

Inverted Back to Original:
 [[1.e+01 5.e+04]
 [2.e+01 7.e+04]
 [3.e+01 9.e+04]]


---
### 2. Vectorized Pairwise Distance Computation (No Loops!)

Given a test matrix $A$ of shape $(M, D)$ and training matrix $B$ of shape $(N, D)$, how do we compute the Euclidean distance between all pairs without $O(M \times N)$ Python loops?

#### The Quadratic Expansion Trick:
$$\|a - b\|^2 = \|a\|^2 - 2 (a \cdot b) + \|b\|^2$$
Using broadcasting:
$$\text{Distances}^2 = \text{norms}_A + \text{norms}_B - 2 A B^T$$

In [12]:
def pairwise_distances_vectorized(A, B):
    # Computes Euclidean distances between M vectors in A and N vectors in B.
    # A: shape (M, D), B: shape (N, D), Returns: shape (M, N) distance matrix
    
    # Sum of squares along feature axis
    A_sq = np.sum(A ** 2, axis=1, keepdims=True)  # Shape (M, 1)
    B_sq = np.sum(B ** 2, axis=1, keepdims=True).T # Shape (1, N)
    
    # Cross product matrix
    cross_term = 2.0 * (A @ B.T)  # Shape (M, N)
    
    # Broadcasted sum
    dist_sq = A_sq + B_sq - cross_term
    
    # Clip negative values caused by floating point precision errors
    dist_sq = np.maximum(dist_sq, 0.0)
    return np.sqrt(dist_sq)

# Performance Benchmark: 200 test samples vs 1,000 training samples
rng = np.random.default_rng(42)
A_test = rng.normal(0, 1, size=(200, 10))
B_train = rng.normal(0, 1, size=(1000, 10))

start = time.time()
dist_matrix = pairwise_distances_vectorized(A_test, B_train)
dur = time.time() - start

print(f"Calculated {dist_matrix.shape[0]} x {dist_matrix.shape[1]} pairwise distances in {dur:.4f} seconds!")
print("Distance matrix shape:", dist_matrix.shape)

Calculated 200 x 1000 pairwise distances in 0.0121 seconds!
Distance matrix shape: (200, 1000)


---
### 3. Mini-Batch Data Generator

In Deep Learning and stochastic optimization, full-batch gradient descent is too slow and memory intensive. We split data into randomized mini-batches.

In [13]:
def get_batches(X, y, batch_size=32, shuffle=True, seed=None):
    # Yields mini-batches of features and labels
    num_samples = len(X)
    rng = np.random.default_rng(seed)
    
    indices = np.arange(num_samples)
    if shuffle:
        rng.shuffle(indices)
        
    for start_idx in range(0, num_samples, batch_size):
        batch_idx = indices[start_idx : start_idx + batch_size]
        yield X[batch_idx], y[batch_idx]

# Demonstrate batch generation
X_dummy = np.arange(100).reshape(50, 2)
y_dummy = np.arange(50)

print("Generating batches of size 16 from 50 samples:")
for batch_num, (xb, yb) in enumerate(get_batches(X_dummy, y_dummy, batch_size=16, shuffle=False)):
    print(f"  Batch {batch_num + 1}: X shape = {xb.shape}, y shape = {yb.shape}")

Generating batches of size 16 from 50 samples:
  Batch 1: X shape = (16, 2), y shape = (16,)
  Batch 2: X shape = (16, 2), y shape = (16,)
  Batch 3: X shape = (16, 2), y shape = (16,)
  Batch 4: X shape = (2, 2), y shape = (2,)


---
### 4. Linear Regression with Batch Gradient Descent

#### Mathematical Formulation:
- **Hypothesis**: $\hat{\mathbf{y}} = X \mathbf{w} + b$
- **Loss Function (MSE)**:
  $$J(\mathbf{w}, b) = \frac{1}{2m} \sum_{i=1}^m (\hat{y}_i - y_i)^2$$
- **Gradients**:
  $$\frac{\partial J}{\partial \mathbf{w}} = \frac{1}{m} X^T (\hat{\mathbf{y}} - \mathbf{y})$$
  $$\frac{\partial J}{\partial b} = \frac{1}{m} \sum_{i=1}^m (\hat{y}_i - y_i)$$

In [14]:
class LinearRegressionScratch:
    def __init__(self, learning_rate=0.01, n_iterations=1000):
        self.lr = learning_rate
        self.n_iterations = n_iterations
        self.weights = None
        self.bias = None
        self.loss_history = []
        
    def fit(self, X, y):
        m, n = X.shape
        self.weights = np.zeros(n)
        self.bias = 0.0
        self.loss_history = []
        
        for i in range(self.n_iterations):
            # 1. Forward pass (predictions)
            y_pred = X @ self.weights + self.bias
            
            # 2. Compute Mean Squared Error
            loss = (1.0 / (2.0 * m)) * np.sum((y_pred - y) ** 2)
            self.loss_history.append(loss)
            
            # 3. Backward pass (analytical gradients)
            error = y_pred - y
            dw = (1.0 / m) * (X.T @ error)
            db = (1.0 / m) * np.sum(error)
            
            # 4. Parameter update step
            self.weights -= self.lr * dw
            self.bias -= self.lr * db
            
        return self
        
    def predict(self, X):
        return X @ self.weights + self.bias

# Synthetic Dataset
rng = np.random.default_rng(42)
X_syn = rng.normal(0, 1, size=(200, 3))
true_w = np.array([2.5, -1.8, 0.7])
true_b = 4.2
y_syn = X_syn @ true_w + true_b + rng.normal(0, 0.2, size=200)

# Train the model
model = LinearRegressionScratch(learning_rate=0.05, n_iterations=300)
model.fit(X_syn, y_syn)

print("True weights:     ", true_w, "| True bias:", true_b)
print("Learned weights:  ", np.round(model.weights, 4), "| Learned bias:", round(model.bias, 4))
print(f"Initial Loss: {model.loss_history[0]:.4f} -> Final Loss: {model.loss_history[-1]:.4f}")

# Compute R2 score
y_pred = model.predict(X_syn)
ss_res = np.sum((y_syn - y_pred) ** 2)
ss_tot = np.sum((y_syn - np.mean(y_syn)) ** 2)
r2_score = 1 - (ss_res / ss_tot)
print(f"Model R-squared (R2) Score: {r2_score:.4f}")

True weights:      [ 2.5 -1.8  0.7] | True bias: 4.2
Learned weights:   [ 2.5074 -1.8015  0.6897] | Learned bias: 4.1926
Initial Loss: 13.8250 -> Final Loss: 0.0205
Model R-squared (R2) Score: 0.9954


---
### 5. Binary Logistic Regression from Scratch

#### Mathematical Formulation:
- **Linear Combination**: $z = X \mathbf{w} + b$
- **Sigmoid Activation**: $\hat{y} = \sigma(z) = \frac{1}{1 + e^{-z}}$
- **Binary Cross-Entropy Loss**:
  $$J(\mathbf{w}, b) = -\frac{1}{m} \sum_{i=1}^m \left[ y_i \ln(\hat{y}_i) + (1 - y_i) \ln(1 - \hat{y}_i) \right]$$
- **Gradients**:
  $$\frac{\partial J}{\partial \mathbf{w}} = \frac{1}{m} X^T (\hat{\mathbf{y}} - \mathbf{y})$$
  $$\frac{\partial J}{\partial b} = \frac{1}{m} \sum_{i=1}^m (\hat{y}_i - y_i)$$

In [15]:
class LogisticRegressionScratch:
    def __init__(self, learning_rate=0.1, n_iterations=500):
        self.lr = learning_rate
        self.n_iterations = n_iterations
        self.weights = None
        self.bias = None
        self.loss_history = []
        
    @staticmethod
    def _sigmoid(z):
        z_safe = np.clip(z, -250.0, 250.0)
        return 1.0 / (1.0 + np.exp(-z_safe))
        
    def fit(self, X, y):
        m, n = X.shape
        self.weights = np.zeros(n)
        self.bias = 0.0
        self.loss_history = []
        
        for _ in range(self.n_iterations):
            # Forward pass
            z = X @ self.weights + self.bias
            y_pred = self._sigmoid(z)
            
            # Loss computation
            eps = 1e-15
            y_pred_clipped = np.clip(y_pred, eps, 1.0 - eps)
            loss = - (1.0 / m) * np.sum(y * np.log(y_pred_clipped) + (1.0 - y) * np.log(1.0 - y_pred_clipped))
            self.loss_history.append(loss)
            
            # Backward pass
            error = y_pred - y
            dw = (1.0 / m) * (X.T @ error)
            db = (1.0 / m) * np.sum(error)
            
            # Update parameters
            self.weights -= self.lr * dw
            self.bias -= self.lr * db
            
        return self
        
    def predict_proba(self, X):
        z = X @ self.weights + self.bias
        return self._sigmoid(z)
        
    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(np.int32)

# Generate synthetic binary classification dataset
rng = np.random.default_rng(101)
X_pos = rng.normal(loc=1.5, scale=0.8, size=(100, 2))
X_neg = rng.normal(loc=-1.5, scale=0.8, size=(100, 2))
X_cls = np.vstack([X_pos, X_neg])
y_cls = np.array([1] * 100 + [0] * 100)

clf = LogisticRegressionScratch(learning_rate=0.2, n_iterations=400)
clf.fit(X_cls, y_cls)

y_pred_cls = clf.predict(X_cls)

# Compute classification evaluation metrics
accuracy = np.mean(y_pred_cls == y_cls)
true_positives = np.sum((y_pred_cls == 1) & (y_cls == 1))
predicted_positives = np.sum(y_pred_cls == 1)
actual_positives = np.sum(y_cls == 1)

precision = true_positives / predicted_positives
recall = true_positives / actual_positives
f1 = 2 * (precision * recall) / (precision + recall)

print(f"Classification Metrics from Scratch:")
print(f"  Accuracy:  {accuracy * 100:.2f}%")
print(f"  Precision: {precision:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"  F1 Score:  {f1:.4f}")

Classification Metrics from Scratch:
  Accuracy:  99.50%
  Precision: 1.0000
  Recall:    0.9900
  F1 Score:  0.9950


---
### 6. Advanced Usages: Ridge Closed-Form, SGD with Momentum, and Multi-Class Softmax

#### A. Closed-Form Ridge Regression with Bias Exemption

In regularized linear regression, we solve:
$$J(w) = \frac{1}{2m} \|X w - y\|^2 + \frac{\lambda}{2} \sum_{j=1}^D w_j^2$$

> **The Intercept Regularization Trap:**
> We must **never penalize the bias intercept term $w_0$**! Penalizing the intercept would force the prediction baseline towards zero, corrupting uncentered datasets.
>
> **The Analytical Solution:**
> $$w = (X_{aug}^T X_{aug} + \lambda I')^{-1} X_{aug}^T y$$
> where $X_{aug} = [1, X]$ and $I'$ is an identity matrix with $I'_{0,0} = 0$.

In [16]:
class RidgeRegressionClosedForm:
    def __init__(self, alpha=1.0):
        self.alpha = alpha
        self.weights_ = None
        
    def fit(self, X, y):
        m, n = X.shape
        # Augment X with column of ones for the bias intercept: shape (m, n + 1)
        X_aug = np.hstack([np.ones((m, 1)), X])
        
        # Identity matrix of size (n + 1, n + 1)
        I_prime = np.eye(n + 1)
        I_prime[0, 0] = 0.0  # Crucial: Do NOT regularize bias intercept!
        
        # Analytical solution: (X_aug.T @ X_aug + alpha * I_prime)^(-1) @ X_aug.T @ y
        A = X_aug.T @ X_aug + self.alpha * I_prime
        b = X_aug.T @ y
        self.weights_ = np.linalg.solve(A, b)
        return self
        
    def predict(self, X):
        X_aug = np.hstack([np.ones((len(X), 1)), X])
        return X_aug @ self.weights_

# Test Ridge with large regularization vs unregularized
ridge_model = RidgeRegressionClosedForm(alpha=10.0)
ridge_model.fit(X_syn, y_syn)

print("Closed-form Ridge Learned Intercept:", round(ridge_model.weights_[0], 4))
print("Closed-form Ridge Learned Weights:  ", np.round(ridge_model.weights_[1:], 4))

Closed-form Ridge Learned Intercept: 4.1997
Closed-form Ridge Learned Weights:   [ 2.3809 -1.7006  0.6351]


#### B. Mini-Batch SGD with Momentum from Scratch

Vanilla Gradient Descent oscillates erratically in steep ravines where the surface curves much more steeply in one dimension than another.
**Momentum** accelerates SGD in the relevant direction and dampens oscillations by maintaining an exponentially decaying moving average of past gradients:
$$v_t = \beta v_{t-1} + (1 - \beta) \nabla_w J$$
$$w \leftarrow w - \alpha v_t$$
where $\beta \approx 0.9$ is the momentum coefficient.

In [17]:
class SGDMomentumLinearRegression:
    def __init__(self, lr=0.01, beta=0.9, batch_size=16, epochs=50):
        self.lr = lr
        self.beta = beta
        self.batch_size = batch_size
        self.epochs = epochs
        self.w = None
        self.b = None
        self.loss_history = []
        
    def fit(self, X, y):
        m, n = X.shape
        self.w = np.zeros(n)
        self.b = 0.0
        v_w = np.zeros(n)
        v_b = 0.0
        self.loss_history = []
        
        for epoch in range(self.epochs):
            # Shuffle indices at the start of each epoch
            indices = np.random.permutation(m)
            X_shuffled = X[indices]
            y_shuffled = y[indices]
            
            for start_idx in range(0, m, self.batch_size):
                xb = X_shuffled[start_idx : start_idx + self.batch_size]
                yb = y_shuffled[start_idx : start_idx + self.batch_size]
                b_size = len(xb)
                
                # Forward pass
                y_pred = xb @ self.w + self.b
                error = y_pred - yb
                
                # Mini-batch gradients
                dw = (1.0 / b_size) * (xb.T @ error)
                db = (1.0 / b_size) * np.sum(error)
                
                # Momentum velocity accumulation
                v_w = self.beta * v_w + (1.0 - self.beta) * dw
                v_b = self.beta * v_b + (1.0 - self.beta) * db
                
                # Update parameters along velocity vector
                self.w -= self.lr * v_w
                self.b -= self.lr * v_b
                
            # Log epoch loss
            epoch_loss = 0.5 * np.mean((X @ self.w + self.b - y) ** 2)
            self.loss_history.append(epoch_loss)
            
        return self

sgd_mom = SGDMomentumLinearRegression(lr=0.05, beta=0.9, batch_size=16, epochs=30)
sgd_mom.fit(X_syn, y_syn)

print("SGD with Momentum Initial Loss:", round(sgd_mom.loss_history[0], 4))
print("SGD with Momentum Final Loss:  ", round(sgd_mom.loss_history[-1], 4))
print("Learned Weights:", np.round(sgd_mom.w, 4), "| Bias:", round(sgd_mom.b, 4))

SGD with Momentum Initial Loss: 7.1158
SGD with Momentum Final Loss:   0.0205
Learned Weights: [ 2.5055 -1.8017  0.6882] | Bias: 4.1945


#### C. Multi-Class Softmax Regression from Scratch

When predicting across $K > 2$ classes:
- **Linear Logits**: $Z = X W + b$ where $W \in \mathbb{R}^{D \times K}$ and $b \in \mathbb{R}^{1 \times K}$.
- **Softmax Activation**: $\hat{Y}_{ik} = \frac{e^{Z_{ik} - \max(Z_i)}}{\sum_c e^{Z_{ic} - \max(Z_i)}}$.
- **Categorical Cross-Entropy Loss**:
  $$J(W, b) = -\frac{1}{m} \sum_{i=1}^m \sum_{k=1}^K Y_{ik} \ln(\hat{Y}_{ik})$$
- **Analytical Gradient Matrix**:
  $$\frac{\partial J}{\partial W} = \frac{1}{m} X^T (\hat{Y} - Y) \quad \in \mathbb{R}^{D \times K}$$
  $$\frac{\partial J}{\partial b} = \frac{1}{m} \sum_{i=1}^m (\hat{Y}_i - Y_i) \quad \in \mathbb{R}^{1 \times K}$$

In [18]:
class SoftmaxRegressionScratch:
    def __init__(self, lr=0.1, n_iterations=600):
        self.lr = lr
        self.n_iterations = n_iterations
        self.W = None
        self.b = None
        self.loss_history = []
        
    @staticmethod
    def _softmax(z):
        max_z = np.max(z, axis=1, keepdims=True)
        exp_z = np.exp(z - max_z)
        return exp_z / np.sum(exp_z, axis=1, keepdims=True)
        
    def fit(self, X, y, num_classes):
        m, n = X.shape
        self.W = np.zeros((n, num_classes))
        self.b = np.zeros((1, num_classes))
        
        # One-Hot Encode target labels: shape (m, num_classes)
        Y_one_hot = np.eye(num_classes)[y]
        self.loss_history = []
        
        for _ in range(self.n_iterations):
            # Forward pass
            Z = X @ self.W + self.b
            Y_pred = self._softmax(Z)
            
            # Cross-Entropy Loss
            eps = 1e-15
            loss = - (1.0 / m) * np.sum(Y_one_hot * np.log(np.clip(Y_pred, eps, 1.0)))
            self.loss_history.append(loss)
            
            # Backward pass (Matrix Gradients)
            error = Y_pred - Y_one_hot
            dW = (1.0 / m) * (X.T @ error)
            db = (1.0 / m) * np.sum(error, axis=0, keepdims=True)
            
            # Gradient descent update
            self.W -= self.lr * dW
            self.b -= self.lr * db
            
        return self
        
    def predict_proba(self, X):
        return self._softmax(X @ self.W + self.b)
        
    def predict(self, X):
        return np.argmax(self.predict_proba(X), axis=1)

# Generate 3-class synthetic classification dataset
rng = np.random.default_rng(202)
c0 = rng.normal(loc=[-2, -2], scale=0.7, size=(80, 2))
c1 = rng.normal(loc=[2, 2], scale=0.7, size=(80, 2))
c2 = rng.normal(loc=[-2, 2], scale=0.7, size=(80, 2))
X_multi = np.vstack([c0, c1, c2])
y_multi = np.array([0] * 80 + [1] * 80 + [2] * 80)

# Train Softmax Regression
multi_clf = SoftmaxRegressionScratch(lr=0.2, n_iterations=400)
multi_clf.fit(X_multi, y_multi, num_classes=3)
y_pred_multi = multi_clf.predict(X_multi)

accuracy = np.mean(y_pred_multi == y_multi) * 100
print(f"Multi-Class Softmax Regression Accuracy from scratch: {accuracy:.2f}%")
print(f"Initial Cross-Entropy: {multi_clf.loss_history[0]:.4f} -> Final Loss: {multi_clf.loss_history[-1]:.4f}")

Multi-Class Softmax Regression Accuracy from scratch: 100.00%
Initial Cross-Entropy: 1.0986 -> Final Loss: 0.0152


### Module 01 Conclusion & Congratulations!
You have successfully mastered **Module 01: NumPy for Machine Learning**!

Throughout these 5 progressive notebooks, you built:
1. Contiguous memory buffers, memory-mapping (`memmap`), and strided windowing (`01_array_basics_and_creation.ipynb`).
2. Multi-dimensional tensor slicing, fancy indexing semantics, and Einstein summation (`02_indexing_slicing_and_reshaping.ipynb`).
3. Vectorization, stable softmax, one-hot encoding, and Mahalanobis distance (`03_vectorization_and_broadcasting.ipynb`).
4. Linear algebra, SVD low-rank compression, Cholesky solvers, and full PCA from scratch (`04_math_stats_and_linear_algebra.ipynb`).
5. Feature scaling, gradient descent, Closed-Form Ridge, SGD with Momentum, and Multi-Class Softmax Regression (`05_practical_ml_applications.ipynb`).

**Up Next:** Proceed to **Module 02: Pandas** to master tabular data exploration, cleaning, aggregation, and feature engineering!